# More plotting with pandas/Matplotlib


### Sources

This lesson is inspired by the [Programming in Python lessons](http://swcarpentry.github.io/python-novice-inflammation/) from the [Software Carpentry organization](http://software-carpentry.org) and has adapted or reused material from University of Helsinki Geo-python course (https://geo-python-site.readthedocs.io/en/latest/course-info/licensing.html ) under a Creative Commons Attribution-ShareAlike 4.0 International licence (https://creativecommons.org/licenses/by-sa/4.0/deed.en). 



At this point you should know the basics of making plots with pandas and the Matplotlib module. Now we will expand on our basic plotting skills to learn how to create more advanced plots. In this part, we will show how to visualize data using pandas/Matplotlib and create multi-panel plots like the ones later in this lesson.



## Input data

In this part of the lesson we'll continue working with the same climate data from the Nordics that we used in the first part of the lecture.


## Getting started

Let's start again by importing the libraries we'll need.

In [ ]:
# pandas for handling the data, matplotlib for the plotting
import pandas as pd
import matplotlib.pyplot as plt

### Loading the data

Now we'll load the data just as we did in the first part of the lesson: 
- Specify the no data values
- Specify a subset of columns
- Parse the `date` column as a datetime index

In [ ]:
# The data file sits in the same folder as this notebook
fp = r"nordics_weather.csv"

In [ ]:
# Read the data, turning 'date' into real dates and using it as the row index
data = pd.read_csv(
    fp,
    na_values=["*", "**", "***", "****", "*****", "******"],
    usecols=["country", "date", "precipitation","snow_depth", "tavg", "tmax", "tmin"],
    parse_dates=["date"],
    index_col="date",
)

In [ ]:
# How many rows did we get?
print(f"Number of rows: {len(data)}")

As you can see, the data set has several thousand rows.

Let's have a closer look at the first rows of data.

In [ ]:
# The first five rows
data.head()

Let's rename the `'tavg'` column as `tavg_F`, since we'll later convert our temperatures from Celsius  to Fahrenheit.

In [ ]:
# Rename 'tavg' to 'tavg_F' using a dictionary of old name: new name
new_names = {"tavg": "tavg_F"}
data = data.rename(columns=new_names)

Check again the first rows of data to confirm successful renaming.

In [ ]:
# Confirm that the column was renamed
data.head()

## Preparing the data

First, we have to deal with no data values. Let's check how many no data values we have.

In [ ]:
# isna() marks every missing value as True, and sum() then counts them per column
print(f"Number of no data values per column:\n{data.isna().sum()}")

The output above shows how many missing values each column has. Let's get rid of the missing values in the `'tavg_F'` column, since that is the column we are going to plot. We need not worry about the no data values in the `'tmax'` and `'tmin'` columns, since we won't be using them for plotting. 

We can remove rows from our DataFrame where `'tavg_F'` is missing using the `dropna()` method, as shown below.

In [ ]:
# inplace=True changes 'data' directly, instead of returning a new DataFrame
data.dropna(subset=["tavg_F"], inplace=True)

In [ ]:
# How many rows are left after dropping the missing values?
print(f"Number of rows after removing no data values: {len(data)}")

> ⚠️ **Common Pitfall: `inplace=True` returns `None`**
>
> There are two ways to apply a method like `dropna()`, and mixing them up is a classic mistake:
>
> ```python
> data = data.dropna(subset=["tavg_F"])                # store the returned copy
> data.dropna(subset=["tavg_F"], inplace=True)         # change data directly
> ```
>
> Both of those work. But combining them throws your data away:
>
> ```python
> data = data.dropna(subset=["tavg_F"], inplace=True)  # WRONG - data is now None!
> ```
>
> Because `inplace=True` returns `None`, assigning the result replaces your DataFrame with nothing. If a DataFrame suddenly becomes `None`, look for this first.

### Check your understanding

What would happen if we removed all rows with any no data values from our data (also considering no data values in the `tmin` and `tmax` columns)?

In [ ]:
# Removing every row with any missing value removes only a few rows here (5,470 of 5,477 remain),
# but in other datasets it can remove a large part of the data.
# Note! Here we are not applying .dropna() "inplace"
#       so we are not making any permanent changes to our DataFrame.
print(f"If we drop all rows with no data values we have {len(data.dropna())} rows left in our DataFrame.")
print(f"If we drop only the missing tavg_F values we have {len(data)} rows in our DataFrame.")

### Converting temperatures to Fahrenheit

Now that we have loaded our data, we can convert the values of temperature from Celsius to Fahrenheit, like we have in earlier lessons.

In [ ]:
# Convert each temperature column from Celsius to Fahrenheit
data["tavg_F"] = (data["tavg_F"] *1.8 ) + 32.0
data["tmax"] = (data["tmax"] *1.8 ) + 32.0
data["tmin"] = (data["tmin"] *1.8 ) + 32.0

Let's check how our dataframe looks like at this point:

In [ ]:
# Check the converted values
data.head()

## Using subplots

Let's continue working with the weather data and learn how to use *subplots*. Subplots are figures where you have multiple plots in different panels of the same figure, as was shown at the start of the lesson.

### Extracting seasonal temperatures

Let's now select data from different seasons of the year in 2017/2018 for Sweden:

- Winter (December 2017 - February 2018)
- Spring (March 2018 - May 2018)
- Summer (June 2018 - August 2018)
- Autumn (September 2018 - November 2018)

In [ ]:
# Filter for Sweden only
data_sweden = data[data["country"] == "Sweden"]

# Select data for each season in 2018
winter = data_sweden.loc[(data_sweden.index >= "2017-12-01") & (data_sweden.index < "2018-03-01")]
winter_temps = winter["tavg_F"]

spring = data_sweden.loc[(data_sweden.index >= "2018-03-01") & (data_sweden.index <= "2018-05-31")]
spring_temps = spring["tavg_F"]

summer = data_sweden.loc[(data_sweden.index >= "2018-06-01") & (data_sweden.index <= "2018-08-31")]
summer_temps = summer["tavg_F"]

autumn = data_sweden.loc[(data_sweden.index >= "2018-09-01") & (data_sweden.index <= "2018-11-30")]
autumn_temps = autumn["tavg_F"]

In [ ]:
# Check one of the seasonal selections
summer_temps.head()

Now we can plot our data to see how the different seasons look separately.

In [ ]:
# Each season plotted on its own - note how the y-axis range differs each time
ax1 = winter_temps.plot()

In [ ]:
ax2 = spring_temps.plot()

In [ ]:
ax3 = summer_temps.plot()

In [ ]:
ax4 = autumn_temps.plot()

OK, so from these plots we can already see that the temperatures in the four seasons are quite different, which is rather obvious of course. It is important to also notice that the scale of the *y*-axis changes in these four plots. If we would like to compare different seasons to each other we need to make sure that the temperature scale is similar in the plots for each season.

### Finding data bounds

Let's set our *y*-axis limits so that the upper limit is the maximum temperature + 5 degrees in our data (full year), and the lowest is the minimum temperature - 5 degrees.

In [ ]:
# Find lower limit for y-axis
min_temp = min(
    winter_temps.min(), spring_temps.min(), summer_temps.min(), autumn_temps.min()
)
min_temp = min_temp - 5.0

# Find upper limit for y-axis
max_temp = max(
    winter_temps.max(), spring_temps.max(), summer_temps.max(), autumn_temps.max()
)
max_temp = max_temp + 5.0

# Print y-axis min, max
print(f"Min: {min_temp}, Max: {max_temp}")

We can now use this temperature range to standardize the y-axis range on our plot.

### Creating our first set of subplots

Let's now continue and see how we can plot all these different plots into the same figure. We can create a 2x2 panel for our visualization using Matplotlib’s `subplots()` function where we specify how many rows and columns we want to have in our figure. We can also specify the size of our figure with `figsize` parameter as we have seen earlier with pandas. As a reminder, `figsize` takes the `width` and `height` values (in inches) as inputs.

In [ ]:
# Create a figure with a 2x2 grid of empty panels (axes)
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 8))
axes

We can see that the result is a 2x2 grid of axes: the first row holds the axes for columns 1 and 2 on **row 1**, and the second holds the axes for columns 1 and 2 on **row 2**. We access an individual panel with `axes[row][column]`.

We can parse these axes into their own variables so it is easier to work with them.

In [ ]:
# Give each panel its own name: ax11 = row 1 column 1, ax12 = row 1 column 2, and so on
ax11 = axes[0][0]
ax12 = axes[0][1]
ax21 = axes[1][0]
ax22 = axes[1][1]

Now we have four axis variables for the different panels in our figure. Next we can use them to plot the seasonal data. Let's begin by plotting the seasons, and give different colors for the lines and specify the *y*-axis range to be the same for all subplots. We can do this using what we know and some parameters below:

- The `c` parameter changes the color of the line. Matplotlib has a [large list of named colors](https://matplotlib.org/stable/gallery/color/named_colors.html) you can consult to customize your color scheme.
- The `lw` parameter controls the width of the lines
- The `ylim` parameter controls the y-axis range

In [ ]:
# Set the plot line width
line_width = 1.5

# Plot data
winter_temps.plot(ax=ax11, c="blue", lw=line_width, ylim=[min_temp, max_temp])
spring_temps.plot(ax=ax12, c="orange", lw=line_width, ylim=[min_temp, max_temp])
summer_temps.plot(ax=ax21, c="green", lw=line_width, ylim=[min_temp, max_temp])
autumn_temps.plot(ax=ax22, c="brown", lw=line_width, ylim=[min_temp, max_temp])

# Display the figure
fig

> ⚠️ **Common Pitfall: forgetting `ax=` when drawing into subplots**
>
> To draw into a particular panel you must tell pandas *which* panel to use, with the `ax=` parameter:
>
> ```python
> winter_temps.plot(ax=ax11, c="blue")   # draws into the top-left panel
> winter_temps.plot(c="blue")            # creates a brand new, separate figure!
> ```
>
> If your subplot grid comes out empty while a stray extra plot appears underneath it, a missing `ax=` is almost always the reason.

Great, now we have all the plots in same figure! However, we can see that there are some problems with our *x*-axis labels and a few missing items we can add. Let's do that below.

In [ ]:
# Create the new figure and subplots
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 8))

# Rename the axes for ease of use
ax11 = axes[0][0]
ax12 = axes[0][1]
ax21 = axes[1][0]
ax22 = axes[1][1]

In [ ]:
# mdates gives us tools for formatting dates on an axis
import matplotlib.dates as mdates


# Set plot line width
line_width = 1.5

# Define the date format for the x-axis labels to show only the month name
date_format = mdates.DateFormatter('%b')  # '%b' for abbreviated month name (Jan, Feb, etc.)

# Plot each season into its own panel using the ax= parameter
winter_temps.plot(ax=ax11, c="blue", lw=line_width, ylim=[min_temp, max_temp], grid=True)
spring_temps.plot(ax=ax12, c="orange", lw=line_width, ylim=[min_temp, max_temp], grid=True)
summer_temps.plot(ax=ax21, c="green", lw=line_width, ylim=[min_temp, max_temp], grid=True)
autumn_temps.plot(ax=ax22, c="brown", lw=line_width, ylim=[min_temp, max_temp], grid=True)

# Apply the date format to the x-axis of each subplot
for ax in [ax11, ax12, ax21, ax22]:
    # Set major locator to month and formatter to abbreviated month name
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(date_format)
    ax.xaxis.set_minor_locator(plt.NullLocator())  # Set minor ticks to NullLocator


# Set figure title
fig.suptitle("2017-2018 seasonal temperature observations for Sweden, in Fahrenheit")


# Axis labels
ax21.set_xlabel("Date")
ax22.set_xlabel("Date")
ax11.set_ylabel("Temperature [°F]")
ax21.set_ylabel("Temperature [°F]")

# Season label text
ax11.set_title("Winter")
ax12.set_title('Spring')
ax21.set_title('Summer')
ax22.set_title('Autumn')


# Fine-tune the spacing so the labels don't overlap
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.05, top=0.9, hspace=0.25, wspace=0.1)

# Display plot
fig


### Check your understanding

Visualize winter and summer temperatures in a 1x2 panel figure.
Save the figure as a .png file.

In [ ]:
# Two subplots side-by-side (1 row, 2 columns)
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12, 4))

# Set plot line width
line_width = 1.5

# Plot data. With a 1-row figure, axs is a simple list: axs[0] and axs[1].
winter_temps.plot(
    ax=axs[0], c="blue", lw=line_width, ylim=[min_temp, max_temp], grid=True
)
summer_temps.plot(
    ax=axs[1], c="green", lw=line_width, ylim=[min_temp, max_temp], grid=True
)

# Set figure title
fig.suptitle(
    "Summer-Winter variations in Sweden"
)
# Apply the date format to the x-axis of each subplot
for ax in axs:
    # Set major locator to month and formatter to abbreviated month name
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(date_format)
    ax.xaxis.set_minor_locator(plt.NullLocator())  # Set minor ticks to NullLocator


# Rotate the x-axis labels so they don't overlap
plt.setp(axs[0].xaxis.get_majorticklabels(), rotation=20)
plt.setp(axs[1].xaxis.get_majorticklabels(), rotation=20)

# Axis labels
axs[0].set_xlabel("Date")
axs[1].set_xlabel("Date")
axs[0].set_ylabel("Temperature [°F]")
axs[1].set_ylabel("Temperature [°F]")

# Season label text
axs[0].set_title("Winter")
axs[1].set_title("Summer")


# Save the figure - in the same cell as the plot
plt.savefig("Summer_Winter.png")

## Extra: pandas/Matplotlib plot style sheets

One cool thing about plotting using pandas/Matplotlib is that is it possible to change the overall style of your plot to one of several available plot style options very easily. Let's consider an example below using the four-panel plot we produced in this lesson.

We will start by defining the plot style, using the `'dark_background'` plot style here.

In [ ]:
# Switch every following plot to the 'dark_background' style.
# This setting stays active until you change it again.
plt.style.use("dark_background")

> ⚠️ **Common Pitfall: `plt.style.use()` changes *every* later plot**
>
> `plt.style.use("dark_background")` is a **global** setting. It doesn't only affect the next plot - every figure you create afterwards in this session will use it, including plots in cells further up if you re-run them.
>
> To go back to the normal look, run:
>
> ```python
> plt.style.use("default")
> ```
>
> If your plots (or your saved image files) mysteriously turn black, this is why.

In [ ]:
# Create the new figure and subplots
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 8))

# Rename the axes for ease of use
ax11 = axes[0][0]
ax12 = axes[0][1]
ax21 = axes[1][0]
ax22 = axes[1][1]

The `plt.style.use()` command above produced no output, but from now on every plot we create will use the `dark_background` styling. Let's see what that looks like.

In [ ]:

# Set plot line width
line_width = 1.5

# Define the date format for the x-axis labels to show only the month name
date_format = mdates.DateFormatter('%b')  # '%b' for abbreviated month name (Jan, Feb, etc.)

# Exactly the same plotting code as before - only the style has changed
winter_temps.plot(ax=ax11, c="blue", lw=line_width, ylim=[min_temp, max_temp], grid=True)
spring_temps.plot(ax=ax12, c="orange", lw=line_width, ylim=[min_temp, max_temp], grid=True)
summer_temps.plot(ax=ax21, c="green", lw=line_width, ylim=[min_temp, max_temp], grid=True)
autumn_temps.plot(ax=ax22, c="brown", lw=line_width, ylim=[min_temp, max_temp], grid=True)

# Apply the date format to the x-axis of each subplot
for ax in [ax11, ax12, ax21, ax22]:
    # Set major locator to month and formatter to abbreviated month name
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(date_format)
    ax.xaxis.set_minor_locator(plt.NullLocator())  # Set minor ticks to NullLocator


# Set figure title
fig.suptitle("2017-2018 seasonal temperature observations for Sweden")


# Axis labels
ax21.set_xlabel("Date")
ax22.set_xlabel("Date")
ax11.set_ylabel("Temperature [°F]")
ax21.set_ylabel("Temperature [°F]")

# Season label text
ax11.set_title("Winter")
ax12.set_title('Spring')
ax21.set_title('Summer')
ax22.set_title('Autumn')


fig.subplots_adjust(left=0.08, right=0.98, bottom=0.01, top=0.9, hspace=0.4, wspace=0.3)

# Display plot
fig


As you can see, the plot format has now changed to use the `dark_background` style. You can find other plot style options in the [complete list of available Matplotlib style sheets](https://matplotlib.org/stable/gallery/style_sheets/style_sheets_reference.html). Have fun!

In [ ]:
import matplotlib.pyplot as plt

# Apply the style - it will set the context for colors and styles
plt.style.use("dark_background")

# Create the figure and subplots
fig, ((ax11, ax12), (ax21, ax22)) = plt.subplots(2, 2)

# Set the figure background color
fig.patch.set_facecolor('black')  # Set the outer color

# Set plot line width
line_width = 1.5

# Plotting data with explicit colors for visibility on dark background
winter_temps.plot(ax=ax11, c="cyan", lw=line_width, ylim=[min_temp, max_temp], grid=True)
spring_temps.plot(ax=ax12, c="yellow", lw=line_width, ylim=[min_temp, max_temp], grid=True)
summer_temps.plot(ax=ax21, c="lime", lw=line_width, ylim=[min_temp, max_temp], grid=True)
autumn_temps.plot(ax=ax22, c="red", lw=line_width, ylim=[min_temp, max_temp], grid=True)

# Set axes background color
for ax in [ax11, ax12, ax21, ax22]:
    ax.set_facecolor('black')  # Set the inner color for each subplot


# Display the plot
plt.show()


## Summary

This lesson built on basic plotting to produce **multi-panel figures**. You can now:

- **Count and remove missing values** with `.isna().sum()` and `.dropna(subset=[...])`.
- **Select subsets by date range** using `.loc[]` with a datetime index - here, one selection per season.
- **Standardise axis ranges** across plots by finding the overall `min()` and `max()` first, so that the panels can be compared fairly.
- **Create a grid of panels** with `plt.subplots(nrows=, ncols=, figsize=)`, and draw into a specific panel with the `ax=` parameter.
- **Customise each panel** with `c=` (colour), `lw=` (line width), `grid=True`, `.set_title()`, `.set_xlabel()`, and `.set_ylabel()`.
- **Tidy up date axes** with `matplotlib.dates` (`MonthLocator`, `DateFormatter`), and adjust spacing with `fig.subplots_adjust()`.
- **Change the whole look** of your figures with `plt.style.use()`.

| Watch out for | Reminder |
| --- | --- |
| `inplace=True` | Returns `None` - never write `data = data.dropna(inplace=True)`. |
| Subplots | Pass `ax=`, or your plot goes into a brand new figure instead of the panel. |
| Comparing panels | Set the same `ylim` on each, or the eye will be misled by different scales. |
| `plt.style.use()` | It is global and sticky - reset it with `plt.style.use("default")`. |

That completes the plotting lessons. You now have the tools to explore a dataset visually and to produce figures good enough for a report.